# AI-Powered Study Assistant (RAG)

A simple Retrieval-Augmented Generation system for Google Colab (T4 GPU). Upload PDF/TXT notes, ask questions, get answers grounded in your own material.

**Model used:** `microsoft/Phi-3-mini-4k-instruct` (4-bit) — small enough to run on a free T4 GPU.

**Pipeline:** Upload -> Extract Text -> Chunk -> Embed -> FAISS Index -> Retrieve -> Generate Answer

Before running, set `Runtime > Change runtime type > T4 GPU`.


## Step 1 — Install Dependencies


In [1]:
# install required packages
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q sentence-transformers faiss-cpu
!pip install -q pypdf
!pip install -q langchain langchain-community
!pip install -q -U langchain-text-splitters
!pip install -q "pillow<12.0,>=8.0"

print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Dependencies installed.


## Step 2 — Imports & GPU Check


In [2]:
import os
import re
import textwrap
import numpy as np
import torch

from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter

if torch.cuda.is_available():
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")

GPU detected: Tesla T4


## Step 3 — Upload Study Materials

Upload one or more PDF or TXT files.


In [3]:
print("Select PDF or TXT files to upload:")
uploaded = files.upload()
print(f"Uploaded {len(uploaded)} file(s): {list(uploaded.keys())}")

Select PDF or TXT files to upload:


Saving Mod 1 PFE.pdf to Mod 1 PFE.pdf
Uploaded 1 file(s): ['Mod 1 PFE.pdf']


## Step 4 — Extract Text

PDF files are parsed with `pypdf`, TXT files are read directly, then whitespace is cleaned up.


In [4]:
def extract_text_from_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"
    return text

def extract_text_from_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

all_documents = {}

for filename in uploaded.keys():
    if filename.lower().endswith(".pdf"):
        raw_text = extract_text_from_pdf(filename)
    elif filename.lower().endswith(".txt"):
        raw_text = extract_text_from_txt(filename)
    else:
        print(f"Skipping unsupported file: {filename}")
        continue

    all_documents[filename] = clean_text(raw_text)
    print(f"{filename}: extracted {len(all_documents[filename])} characters")

print(f"Extracted text from {len(all_documents)} document(s).")

Mod 1 PFE.pdf: extracted 4838 characters
Extracted text from 1 document(s).


## Step 5 — Chunk the Text

Text is split into overlapping chunks (900 chars, 150 overlap) so the embedding model and LLM get focused pieces of context instead of whole documents.


In [5]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

corpus_chunks = []
chunk_metadata = []

for filename, text in all_documents.items():
    doc_chunks = splitter.split_text(text)
    for i, c in enumerate(doc_chunks):
        if len(c.strip()) < 30:
            continue
        corpus_chunks.append(c)
        chunk_metadata.append({"source": filename, "chunk_id": i})

print(f"Created {len(corpus_chunks)} chunks from {len(all_documents)} document(s).")
print("Example chunk:\n", textwrap.fill(corpus_chunks[0], 100))

Created 7 chunks from 1 document(s).
Example chunk:
 Mod 1 1. Introduction to Philosophy Philosophy is the systematic study of existence, knowledge, and
values using human reason. • Etymology: Derived from Greek words Philia (Love) and Sophia (Wisdom).
It is the "Love of Wisdom." • Nature: o Science: It studies beings (all things that exist) in their
ultimate causes through reason. o Art: It involves the creative and critical application of
knowledge to life’s purpose. • Origin: * Ancient: Triggered by Wonder (curiosity about the
universe). o Modern: Triggered by Doubt (Descartes started with doubt to find certain truth).
Example: While a biologist asks how a heart beats (scientific), a philosopher asks why life exists
at all or what makes a life "good" (philosophical). 2. Branches of Philosophy Philosophy is vast, so
it is divided into specific areas of study. A


## Step 6 — Embeddings & FAISS Index

Chunks are embedded with `BAAI/bge-base-en-v1.5` and stored in a FAISS index for similarity search. The index is also saved to disk so it doesn't need to be rebuilt every session.


In [6]:
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

FAISS_INDEX_PATH = "study_assistant.index"
CHUNKS_CACHE_PATH = "study_assistant_chunks.npy"

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")

print("Encoding chunks...")
chunk_embeddings = embed_model.encode(
    corpus_chunks,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embedding_dim = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(chunk_embeddings)
print(f"FAISS index built with {faiss_index.ntotal} vectors (dim={embedding_dim}).")

# save so we don't have to re-embed next time
faiss.write_index(faiss_index, FAISS_INDEX_PATH)
np.save(CHUNKS_CACHE_PATH, np.array(corpus_chunks, dtype=object), allow_pickle=True)
print("Index saved to disk.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index built with 7 vectors (dim=768).
Index saved to disk.


## Step 7 — Retrieval

Embed the query, search FAISS for candidates, then re-rank them with a cross-encoder for better precision before returning the top matches.


In [7]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda" if torch.cuda.is_available() else "cpu")

def retrieve_relevant_chunks(query, top_k=3, fetch_k=8):
    prefixed_query = BGE_QUERY_PREFIX + query
    query_embedding = embed_model.encode([prefixed_query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = faiss_index.search(query_embedding, fetch_k)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        candidates.append({
            "text": corpus_chunks[idx],
            "score": float(score),
            "source": chunk_metadata[idx]["source"],
        })

    if not candidates:
        return []

    # re-rank with cross-encoder for better precision
    pairs = [[query, c["text"]] for c in candidates]
    rerank_scores = reranker.predict(pairs)
    for c, rs in zip(candidates, rerank_scores):
        c["rerank_score"] = float(rs)

    candidates.sort(key=lambda c: c["rerank_score"], reverse=True)
    return candidates[:top_k]

# quick test
for r in retrieve_relevant_chunks("What is this document about?", top_k=3):
    print(f"[{r['source']} | cosine={r['score']:.3f}]")
    print(textwrap.fill(r["text"], 100))
    print("-" * 80)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[Mod 1 PFE.pdf | cosine=0.473]
. 3. Ethics & Moral Philosophy Ethics is the branch of philosophy that deals with human conduct and
serves as a guide for decision-making. • Nature of Moral Judgments: It is a mental act of evaluating
an action as "Right" or "Wrong" based on a standard of "Good." • Role of Emotions: Emotions play a
critical role in how we react to moral situations (e.g., feeling guilt after lying). • Sources of
Morality: Where do we get our "rules"? o Family/Parents o Society/Culture o Religion/Beliefs o Peers
and Social Norms 4. Core Theories of Ethics These are the frameworks used to solve ethical dilemmas.
(Very important for Section B and Case Studies). A. Utilitarianism (Outcome-Based) • Concept:
Founded on Consequentialism. The "right" action is the one that produces the greatest good for the
greatest number of people. • Focus: Outcomes and results, not the intent
--------------------------------------------------------------------------------
[Mod 1 PFE.pdf | cosin

## Step 8 — Load the LLM

Phi-3-mini-4k-instruct is loaded in 4-bit precision so it fits on a T4's VRAM.


In [8]:
LLM_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=False,
)

generator = pipeline("text-generation", model=llm_model, tokenizer=tokenizer)
print("LLM loaded.")

Loading model...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

LLM loaded.


## Step 9 — Prompt & Answer Generation

The system prompt tells the model to answer only from the retrieved context, and to say so clearly if the answer isn't in the material (this avoids hallucinated answers).


In [9]:
def build_rag_prompt(question, retrieved_chunks):
    context_text = "\n\n".join(f"[Source: {c['source']}]\n{c['text']}" for c in retrieved_chunks)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a study assistant. Answer ONLY using the context below. "
                "Do NOT use any outside knowledge, even if you know the answer yourself.\n\n"
                "Example:\n"
                "Context: [about machine learning]\n"
                "Question: What is the capital of France?\n"
                "Answer: I cannot find this information in the provided materials.\n\n"
                "If the context does not contain the answer, respond with EXACTLY: "
                "'I cannot find this information in the provided materials.' Nothing else."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context_text}\n\nQuestion: {question}",
        },
    ]
    return messages

def generate_answer(question, top_k=3, max_new_tokens=300, do_sample=True):
    retrieved_chunks = retrieve_relevant_chunks(question, top_k=top_k)
    if not retrieved_chunks:
        return "No relevant content found. Please upload study materials first.", []

    messages = build_rag_prompt(question, retrieved_chunks)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "repetition_penalty": 1.1,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        generation_kwargs["temperature"] = 0.15
        generation_kwargs["top_p"] = 0.95

    output = generator(prompt, **generation_kwargs)
    full_text = output[0]["generated_text"]
    answer = full_text[len(prompt):].strip()
    return answer, retrieved_chunks

## Step 10 — Ask a Question


In [10]:
question = "Summarize the document."
answer, sources_used = generate_answer(question, top_k=3)

print("Question:", question)
print("\nAnswer:\n", textwrap.fill(answer, 100))
print("\nSources used:")
for s in sources_used:
    print(f"  - {s['source']} (similarity={s['score']:.3f})")

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'top_p', 'temperature', 'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is 

Question: Summarize the document.

Answer:
 This text provides insights into various philosophical concepts related to morality and ethics
within different branches of philosophy. In terms of ethics, there's discussion around nature of
moral judgments which involve evaluation against standards like “Good,” highlighting emotions such
as guilt when acting immorally, sources where these rules originate including family, culture,
religion, peers, etc. Key theories include utilitarianism focusing on outcome maximization ("the
greatest good for the greatest number"), contrasted with other approaches prioritizing individual
rights protection under negative and positive rights respectively; lastly casuistry uses case
reasoning analogies. Additionally, introductory aspects define philosophy broadly concerning its
love for wisdom derived etymologically from ancient Greece, differentiated between scientific
inquiry regarding natural phenomena versus artistic pursuits aimed towards understanding l

## Step 11 — Interactive Q&A (Optional)

Uncomment and run to chat with the assistant. Type `exit` to stop.


In [11]:
# while True:
#     user_q = input("Ask a question (or type 'exit'): ")
#     if user_q.strip().lower() == "exit":
#         print("Goodbye!")
#         break
#     ans, srcs = generate_answer(user_q, top_k=3)
#     print("\nAnswer:\n", textwrap.fill(ans, 100))
#     print("Sources:", [s["source"] for s in srcs])

## Step 12 — Evaluation

The system is evaluated on two sets of questions:
- **In-scope questions** (factual, summarization, synthesis, detail) — scored on retrieval similarity and groundedness.
- **Refusal check** (out-of-scope / trick questions) — checks whether the assistant correctly declines instead of hallucinating an answer.

Groundedness is measured two ways: lexical overlap (word matching) and semantic similarity (embedding cosine similarity), since lexical overlap alone unfairly penalizes correct paraphrasing.


In [12]:
import time
import string

REFUSAL_PHRASE = "cannot find this information"

def word_overlap_score(answer, context_chunks):
    """Fraction of answer words that also appear in the retrieved context."""
    stopwords = {
        "the","a","an","is","are","was","were","of","to","and","in","on",
        "for","with","this","that","it","as","by","be","at","or","from"
    }

    def normalize(text):
        text = text.lower().translate(str.maketrans("", "", string.punctuation))
        return set(w for w in text.split() if w not in stopwords and len(w) > 2)

    answer_words = normalize(answer)
    context_words = set()
    for c in context_chunks:
        context_words |= normalize(c["text"])

    if not answer_words:
        return 0.0
    return len(answer_words & context_words) / len(answer_words)

def embedding_groundedness_score(answer, context_chunks):
    """Cosine similarity between the answer and the retrieved context (embedding-based)."""
    if not answer.strip() or not context_chunks:
        return 0.0
    context_text = " ".join(c["text"] for c in context_chunks)
    pair_embeddings = embed_model.encode([answer, context_text], convert_to_numpy=True, normalize_embeddings=True)
    return float(np.dot(pair_embeddings[0], pair_embeddings[1]))

def evaluate_query(question, top_k=3):
    start_time = time.time()
    answer, retrieved = generate_answer(question, top_k=top_k, do_sample=False)
    latency = time.time() - start_time

    avg_cosine_similarity = np.mean([r["score"] for r in retrieved]) if retrieved else 0.0
    lexical_groundedness = word_overlap_score(answer, retrieved)
    semantic_groundedness = embedding_groundedness_score(answer, retrieved)
    correctly_refused = REFUSAL_PHRASE.lower() in answer.lower()

    return {
        "question": question,
        "answer": answer,
        "avg_cosine_similarity": round(float(avg_cosine_similarity), 3),
        "lexical_groundedness": round(lexical_groundedness, 3),
        "semantic_groundedness": round(semantic_groundedness, 3),
        "answer_length_words": len(answer.split()),
        "latency_seconds": round(latency, 2),
        "correctly_refused": correctly_refused,
    }

sample_questions = [
    ("Factual", "What is the main topic of the notes?"),
    ("Factual", "Define the key term introduced early in the document."),
    ("Summarization", "Summarize the key concepts covered in these notes."),
    ("Summarization", "What are the main takeaways a student should remember?"),
    ("Synthesis", "How do the different concepts in this document relate to each other?"),
    ("Synthesis", "Compare two ideas discussed in the material, if applicable."),
    ("Detail", "Give a specific example or detail mentioned in the notes."),
]

refusal_questions = [
    ("Out-of-scope", "What is the capital of Australia?"),
    ("Out-of-scope", "What is the boiling point of water at sea level?"),
    ("Out-of-scope", "Who won the FIFA World Cup in 2022?"),
    ("Adjacent-absent", "What does the document say about quantum computing?"),
    ("Trick", "What was the author's opinion on the failure of the third proposed method?"),
    ("Fabricated-term", "Explain the 'Meridian Convergence Algorithm' described in these notes."),
]

print("Running in-scope evaluation...\n")
eval_results = [evaluate_query(q) for _, q in sample_questions]
for (category, _), r in zip(sample_questions, eval_results):
    r["category"] = category

print("Running refusal check...\n")
refusal_results = [evaluate_query(q) for _, q in refusal_questions]
for (category, _), r in zip(refusal_questions, refusal_results):
    r["category"] = category

for r in eval_results:
    print(f"[{r['category']}] Q: {r['question']}")
    print(f"  Avg. Cosine Similarity  : {r['avg_cosine_similarity']}")
    print(f"  Lexical Groundedness    : {r['lexical_groundedness']}")
    print(f"  Semantic Groundedness   : {r['semantic_groundedness']}")
    print(f"  Answer Length (words)   : {r['answer_length_words']}")
    print(f"  Latency (s)             : {r['latency_seconds']}")
    print("-" * 80)

avg_cos = np.mean([r["avg_cosine_similarity"] for r in eval_results])
avg_lex_ground = np.mean([r["lexical_groundedness"] for r in eval_results])
avg_sem_ground = np.mean([r["semantic_groundedness"] for r in eval_results])
avg_latency = np.mean([r["latency_seconds"] for r in eval_results])

print(f"\nIn-Scope Overall -> Cosine Sim: {avg_cos:.3f} | "
      f"Lexical Groundedness: {avg_lex_ground:.3f} | "
      f"Semantic Groundedness: {avg_sem_ground:.3f} | "
      f"Latency: {avg_latency:.2f}s")

print("\n" + "=" * 80)
print("REFUSAL CHECK (out-of-scope questions — should be declined, not answered)")
print("=" * 80)
for r in refusal_results:
    status = "Correctly refused" if r["correctly_refused"] else "Did NOT refuse (possible hallucination)"
    print(f"[{r['category']}] Q: {r['question']}")
    print(f"  Answer   : {r['answer']}")
    print(f"  Status   : {status}")
    print(f"  Latency  : {r['latency_seconds']}s")
    print("-" * 80)

refusal_rate = np.mean([r["correctly_refused"] for r in refusal_results])
print(f"\nRefusal Accuracy: {refusal_rate:.0%} ({sum(r['correctly_refused'] for r in refusal_results)}/{len(refusal_results)})")

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'repetition_penalty', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running in-scope evaluation...



[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Running refusal check...



[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been 

[Factual] Q: What is the main topic of the notes?
  Avg. Cosine Similarity  : 0.421
  Lexical Groundedness    : 0.362
  Semantic Groundedness   : 0.837
  Answer Length (words)   : 99
  Latency (s)             : 11.47
--------------------------------------------------------------------------------
[Factual] Q: Define the key term introduced early in the document.
  Avg. Cosine Similarity  : 0.442
  Lexical Groundedness    : 1.0
  Semantic Groundedness   : 0.587
  Answer Length (words)   : 3
  Latency (s)             : 1.11
--------------------------------------------------------------------------------
[Summarization] Q: Summarize the key concepts covered in these notes.
  Avg. Cosine Similarity  : 0.47
  Lexical Groundedness    : 0.193
  Semantic Groundedness   : 0.731
  Answer Length (words)   : 109
  Latency (s)             : 12.92
--------------------------------------------------------------------------------
[Summarization] Q: What are the main takeaways a student should remember?

## Summary

A working RAG study assistant that ingests PDF/TXT notes, retrieves relevant chunks with FAISS + a cross-encoder re-ranker, and generates grounded answers with Phi-3-mini. Evaluation covers both answer quality (cosine similarity, groundedness) and robustness (refusal on out-of-scope questions).